# PTB-XL: ECG + Wavelet Features with RBF SVM

This notebook compares:

1. **12-lead baseline:** ECG and wavelet features concatenated from all 12 leads.
2. **12 single-lead models:** the same feature extractor and classifier, trained independently on each lead.

The target is the five PTB-XL diagnostic superclasses: `NORM`, `MI`, `STTC`, `CD`, and `HYP`. This remains a **multi-label** problem: a recording can have more than one positive label.

## Evaluation protocol

- Folds **1–8**: training.
- Fold **9**: hyperparameter and decision-threshold selection.
- Fold **10**: final test evaluation.

The best single lead is selected using **validation macro-F1**, not test performance. Test metrics are then reported for every lead and the 12-lead baseline. This avoids using the test fold to select the winner.

All features are computed independently from each lead. No cross-lead feature or information from the other 11 leads is used in any single-lead experiment.

> The R-peak detector below is a lightweight feature-engineering aid, not a clinically validated ECG delineation algorithm. Intervals derived from it should be interpreted as approximate signal features.

In [4]:
# If a dependency is missing, run this in a notebook cell:
%pip install wfdb PyWavelets scipy scikit-learn matplotlib pandas tqdm

from pathlib import Path
import ast
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd



import pywt
print(pywt.__version__)
import wfdb
from scipy.signal import butter, sosfiltfilt, find_peaks, peak_widths
from scipy.stats import kurtosis, skew
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    classification_report,
    f1_score,
    hamming_loss,
    multilabel_confusion_matrix,
    roc_auc_score,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

warnings.filterwarnings("ignore", category=RuntimeWarning)

Note: you may need to restart the kernel to use updated packages.


error: externally-managed-environment

× This environment is externally managed
╰─> This Python installation is managed by uv and should not be modified.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.


ModuleNotFoundError: No module named 'pywt'

In [ ]:
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

# Automatically locate the project root (expects data/ptb-xl below it).
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "data" / "ptb-xl").is_dir():
        PROJECT = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find data/ptb-xl. Open this notebook from your project, "
        "or update PROJECT manually."
    )

DATA = next((PROJECT / "data" / "ptb-xl").rglob("ptbxl_database.csv")).parent
SCP_FILE = DATA / "scp_statements.csv"

CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
N_LEADS = len(LEAD_NAMES)
EXPECTED_FS = 100  # filename_lr is the 100 Hz PTB-XL version

# The same small tuning grid is used for every experiment. Expand these lists
# for a more exhaustive search after the first complete run.
C_VALUES = [1, 5, 10]
GAMMA_VALUES = ["scale"]
THRESHOLDS = [-0.5, -0.25, 0.0, 0.1, 0.25, 0.5, 0.75]

# n_jobs=-1 parallels the five one-vs-rest classifiers. Set to 1 if RAM is tight.
OVR_N_JOBS = -1

FEATURE_VERSION = "ecg_wavelet_v2"
CACHE_PATH = DATA / f"ptbxl_{FEATURE_VERSION}_features.npz"

print("Project:", PROJECT)
print("Dataset:", DATA)
print("Feature cache:", CACHE_PATH)
print("SVM grid:", {"C": C_VALUES, "gamma": GAMMA_VALUES})

## 1. Load labels and patient-wise folds

This section intentionally uses the same PTB-XL label mapping and folds as the original notebook. The feature cache is validated against the current record order, labels, folds, and feature version.

In [ ]:
df = pd.read_csv(DATA / "ptbxl_database.csv")
df["scp_codes"] = df["scp_codes"].apply(ast.literal_eval)
scp = pd.read_csv(SCP_FILE, index_col=0)
scp = scp[scp["diagnostic"] == 1]

def make_labels(codes):
    found = {
        scp.at[code, "diagnostic_class"]
        for code in codes
        if code in scp.index and pd.notna(scp.at[code, "diagnostic_class"])
    }
    return [int(label in found) for label in CLASSES]

Y = np.asarray(df["scp_codes"].apply(make_labels).tolist(), dtype=np.uint8)
folds = df["strat_fold"].to_numpy()
record_names = df["filename_lr"].astype(str).to_numpy()

train_mask = folds <= 8
val_mask = folds == 9
test_mask = folds == 10

if not (train_mask.any() and val_mask.any() and test_mask.any()):
    raise ValueError("Expected PTB-XL strat_fold values 1-10.")

print("Records:", len(df))
print("Train / validation / test:", train_mask.sum(), val_mask.sum(), test_mask.sum())
print("Class positives in training:", dict(zip(CLASSES, Y[train_mask].sum(axis=0))))

## 2. ECG and wavelet feature extraction

Each lead is transformed independently into one feature vector (62 features per lead).

**ECG/signal features** include robust amplitude and distribution statistics, signal variation, approximate R-peak amplitude/width, RR-interval statistics, and summaries of a median R-aligned beat in windows before, around, and after QRS.

**Wavelet features** use a level-4 Daubechies-4 discrete wavelet decomposition. For the approximation and detail coefficient arrays, the notebook calculates coefficient spread, mean absolute magnitude, energy proportion, and normalized entropy.

The same feature names and ordering are used for every lead. The 12-lead baseline concatenates the 12 per-lead vectors; a single-lead experiment uses only one vector.

In [ ]:
def _safe_float(value, default=0.0):
    value = float(value)
    return value if np.isfinite(value) else float(default)


def _coefficient_entropy(coeff):
    energy = np.square(np.asarray(coeff, dtype=np.float64))
    total = energy.sum()
    if total <= 0 or not np.isfinite(total):
        return 0.0
    probabilities = energy / total
    probabilities = probabilities[probabilities > 0]
    return _safe_float(-np.sum(probabilities * np.log(probabilities + 1e-12)))


def feature_names_for_lead():
    names = [
        "mean", "std", "rms", "median", "mad", "min", "max", "peak_to_peak",
        "q10", "q25", "q75", "q90", "skew", "kurtosis",
        "mean_abs_diff", "std_diff", "line_length", "zero_crossing_rate",
        "rr_mean_s", "rr_std_s", "rr_median_s", "rr_iqr_s", "rr_min_s", "rr_max_s",
        "rmssd_s", "mean_heart_rate_bpm",
        "peak_abs_median", "peak_abs_iqr", "peak_prominence_median", "peak_width_median_s",
        "pre_qrs_mean", "pre_qrs_std", "pre_qrs_min", "pre_qrs_max",
        "qrs_window_mean", "qrs_window_std", "qrs_window_min", "qrs_window_max",
        "post_qrs_mean", "post_qrs_std", "post_qrs_min", "post_qrs_max",
    ]
    for coeff_name in ["cA4", "cD4", "cD3", "cD2", "cD1"]:
        names.extend([
            f"{coeff_name}_std", f"{coeff_name}_mean_abs",
            f"{coeff_name}_energy_ratio", f"{coeff_name}_entropy",
        ])
    return names


FEATURE_NAMES = feature_names_for_lead()


def extract_ecg_wavelet_features(signal, fs=100):
    """Extract a fixed-length feature vector from one ECG lead (one recording)."""
    x = np.asarray(signal, dtype=np.float64).reshape(-1)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    if x.size < 64:
        raise ValueError(f"Signal is too short for feature extraction: {x.size} samples")

    # Bandpass for feature measurements. A separate 5-20 Hz copy is used only
    # for coarse QRS/R-peak candidate detection.
    nyquist = fs / 2.0
    high = min(40.0, nyquist * 0.90)
    low = min(0.5, high * 0.25)
    sos = butter(3, [low, high], btype="bandpass", fs=fs, output="sos")
    filtered = sosfiltfilt(sos, x)
    centered = filtered - np.median(filtered)

    diff = np.diff(centered)
    abs_diff = np.abs(diff)
    std = np.std(centered)
    q10, q25, q75, q90 = np.quantile(centered, [0.10, 0.25, 0.75, 0.90])
    crossings = np.mean((centered[:-1] >= 0) != (centered[1:] >= 0))

    values = [
        np.mean(centered),
        std,
        np.sqrt(np.mean(np.square(centered))),
        np.median(centered),
        np.median(np.abs(centered - np.median(centered))),
        np.min(centered),
        np.max(centered),
        np.ptp(centered),
        q10, q25, q75, q90,
        skew(centered, bias=False),
        kurtosis(centered, fisher=True, bias=False),
        np.mean(abs_diff),
        np.std(diff),
        np.sum(abs_diff) / max(1, len(centered) - 1),
        crossings,
    ]

    # Coarse R-peak candidates on a QRS-emphasized signal. RR values are
    # approximate and should not be interpreted as clinical measurements.
    det_low, det_high = 5.0, min(20.0, nyquist * 0.90)
    detector = butter(2, [det_low, det_high], btype="bandpass", fs=fs, output="sos")
    qrs_signal = np.abs(sosfiltfilt(detector, centered))
    prominence = max(float(np.std(qrs_signal) * 0.35), 1e-8)
    peaks, peak_properties = find_peaks(
        qrs_signal,
        distance=max(1, int(0.30 * fs)),
        prominence=prominence,
    )

    rr = np.diff(peaks) / float(fs) if len(peaks) >= 2 else np.asarray([], dtype=float)
    if rr.size:
        rr_mean = np.mean(rr)
        rr_std = np.std(rr)
        rr_median = np.median(rr)
        rr_iqr = np.quantile(rr, 0.75) - np.quantile(rr, 0.25)
        rr_min, rr_max = np.min(rr), np.max(rr)
        rmssd = np.sqrt(np.mean(np.square(np.diff(rr)))) if rr.size >= 2 else 0.0
        mean_hr = 60.0 / rr_mean if rr_mean > 0 else 0.0
    else:
        rr_mean = rr_std = rr_median = rr_iqr = rr_min = rr_max = rmssd = mean_hr = 0.0

    peak_amplitudes = qrs_signal[peaks] if len(peaks) else np.asarray([], dtype=float)
    prominences = peak_properties.get("prominences", np.asarray([], dtype=float))
    if len(peaks):
        widths = peak_widths(qrs_signal, peaks, rel_height=0.5)[0] / float(fs)
    else:
        widths = np.asarray([], dtype=float)

    def median_or_zero(array):
        return float(np.median(array)) if len(array) else 0.0

    def iqr_or_zero(array):
        return float(np.quantile(array, 0.75) - np.quantile(array, 0.25)) if len(array) else 0.0

    values.extend([
        rr_mean, rr_std, rr_median, rr_iqr, rr_min, rr_max, rmssd, mean_hr,
        median_or_zero(peak_amplitudes), iqr_or_zero(peak_amplitudes),
        median_or_zero(prominences), median_or_zero(widths),
    ])

    # Build a median R-aligned beat template and summarize approximate windows.
    # The windows are intentionally named by their position relative to QRS;
    # they are not a clinical P/QRS/T-wave delineation.
    pre_samples = int(round(0.22 * fs))
    post_samples = int(round(0.38 * fs))
    beat_segments = [
        centered[p - pre_samples:p + post_samples]
        for p in peaks
        if p >= pre_samples and p + post_samples <= len(centered)
    ]
    if beat_segments:
        median_beat = np.median(np.vstack(beat_segments), axis=0)
        beat_times = (np.arange(len(median_beat)) - pre_samples) / float(fs)
        morphology_windows = [
            ("pre_qrs", (beat_times >= -0.20) & (beat_times < -0.08)),
            ("qrs_window", (beat_times >= -0.06) & (beat_times <= 0.06)),
            ("post_qrs", (beat_times >= 0.10) & (beat_times <= 0.30)),
        ]
        for _, mask in morphology_windows:
            segment = median_beat[mask]
            if segment.size:
                values.extend([np.mean(segment), np.std(segment), np.min(segment), np.max(segment)])
            else:
                values.extend([0.0, 0.0, 0.0, 0.0])
    else:
        values.extend([0.0] * 12)

    # DWT: pywt.wavedec order is [cA4, cD4, cD3, cD2, cD1].
    wavelet = "db4"
    level = min(4, pywt.dwt_max_level(len(centered), pywt.Wavelet(wavelet).dec_len))
    coeffs = pywt.wavedec(centered, wavelet, mode="symmetric", level=level)
    # Pad missing high-level arrays defensively if a shorter signal is supplied.
    coeff_names = [f"cA{level}"] + [f"cD{k}" for k in range(level, 0, -1)]
    total_energy = sum(float(np.sum(np.square(c))) for c in coeffs) + 1e-12
    for coeff in coeffs:
        values.extend([
            np.std(coeff),
            np.mean(np.abs(coeff)),
            np.sum(np.square(coeff)) / total_energy,
            _coefficient_entropy(coeff),
        ])

    # This project uses PTB-XL's 100 Hz low-resolution records. The chosen
    # level is therefore 4, yielding exactly the 20 wavelet features above.
    if len(values) != len(FEATURE_NAMES):
        raise ValueError(
            f"Feature length mismatch ({len(values)} vs {len(FEATURE_NAMES)}). "
            f"DWT level was {level}; expected level 4 at {fs} Hz."
        )
    return np.nan_to_num(np.asarray(values, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)

## 3. Extract features or load the cache

The feature array has shape:

`(number_of_records, 12, number_of_features_per_lead)`

Feature extraction is done once and cached as a compressed `.npz` file beside the PTB-XL metadata. Re-running the notebook can then skip waveform processing.

In [ ]:
cache_is_valid = False
features = None

if CACHE_PATH.exists():
    try:
        with np.load(CACHE_PATH, allow_pickle=False) as cached:
            cache_is_valid = (
                str(cached["version"].item()) == FEATURE_VERSION
                and np.array_equal(cached["record_names"], record_names)
                and np.array_equal(cached["labels"], Y)
                and np.array_equal(cached["folds"], folds)
                and np.array_equal(cached["feature_names"], np.asarray(FEATURE_NAMES))
                and cached["features"].shape == (len(df), N_LEADS, len(FEATURE_NAMES))
            )
            if cache_is_valid:
                features = cached["features"].astype(np.float32, copy=True)
                print("Loaded cached features:", features.shape)
            else:
                print("Existing cache does not match this dataset/configuration; rebuilding.")
    except Exception as exc:
        print("Could not use existing feature cache; rebuilding:", repr(exc))

if not cache_is_valid:
    start = time.perf_counter()
    all_features = np.empty((len(df), N_LEADS, len(FEATURE_NAMES)), dtype=np.float32)
    observed_fs = None
    observed_shape = None

    for row_idx, filename in enumerate(tqdm(record_names, desc="Extracting per-lead features")):
        signal, fields = wfdb.rdsamp(str(DATA / filename))
        signal = np.asarray(signal, dtype=np.float32)
        fs = float(fields.get("fs", EXPECTED_FS))

        if fs != EXPECTED_FS:
            raise ValueError(f"Expected {EXPECTED_FS} Hz, but {filename} reports {fs} Hz")
        if signal.ndim != 2 or signal.shape[1] != N_LEADS:
            raise ValueError(f"Expected (samples, 12) waveform for {filename}; got {signal.shape}")
        if observed_shape is None:
            observed_shape, observed_fs = signal.shape, fs
        elif signal.shape != observed_shape:
            raise ValueError(f"Inconsistent waveform shape for {filename}: {signal.shape} vs {observed_shape}")

        for lead_idx in range(N_LEADS):
            try:
                all_features[row_idx, lead_idx] = extract_ecg_wavelet_features(signal[:, lead_idx], fs=fs)
            except Exception as exc:
                raise RuntimeError(
                    f"Feature extraction failed for record {filename}, lead {LEAD_NAMES[lead_idx]}"
                ) from exc

    features = np.nan_to_num(all_features, nan=0.0, posinf=0.0, neginf=0.0)
    np.savez_compressed(
        CACHE_PATH,
        features=features,
        labels=Y,
        folds=folds,
        record_names=record_names,
        feature_names=np.asarray(FEATURE_NAMES),
        version=np.asarray(FEATURE_VERSION),
    )
    print("Saved feature cache:", CACHE_PATH)
    print("Feature extraction time: {:.1f} minutes".format((time.perf_counter() - start) / 60))

print("Feature tensor:", features.shape)
print("Features per lead:", len(FEATURE_NAMES))
print("All finite:", np.isfinite(features).all())

## 4. Create the 12-lead and single-lead data matrices

The 12-lead baseline concatenates the feature vectors for leads I, II, III, aVR, aVL, aVF, V1–V6. Each single-lead model sees only its corresponding feature vector.

In [ ]:
# Split labels once; every experiment uses identical records and folds.
Y_train, Y_val, Y_test = Y[train_mask], Y[val_mask], Y[test_mask]

# All-lead features: (records, 12 * features_per_lead)
X_12lead = features.reshape(len(features), -1)
X_12_train = X_12lead[train_mask]
X_12_val = X_12lead[val_mask]
X_12_test = X_12lead[test_mask]

# Per-lead features: each item has shape (records, features_per_lead)
X_single = {
    lead_name: features[:, lead_idx, :]
    for lead_idx, lead_name in enumerate(LEAD_NAMES)
}

print("12-lead matrix:", X_12lead.shape)
print("Single-lead matrix:", next(iter(X_single.values())).shape)
print("Train / validation / test:", X_12_train.shape, X_12_val.shape, X_12_test.shape)

## 5. Train and tune an RBF one-vs-rest SVM

For every experiment:

1. Fit `StandardScaler` on training data only.
2. Train one RBF SVM per superclass using one-vs-rest classification and balanced class weights.
3. Select `C`, `gamma`, and one global decision threshold using validation macro-F1.
4. Evaluate the selected configuration once on fold 10.

The same grid and threshold candidates are used for the 12-lead model and all 12 single-lead models.

In [ ]:
def safe_macro_auc(y_true, scores):
    per_class = []
    for class_idx in range(y_true.shape[1]):
        if np.unique(y_true[:, class_idx]).size < 2:
            continue
        per_class.append(roc_auc_score(y_true[:, class_idx], scores[:, class_idx]))
    return float(np.mean(per_class)) if per_class else float("nan")


def safe_macro_ap(y_true, scores):
    per_class = []
    for class_idx in range(y_true.shape[1]):
        if y_true[:, class_idx].sum() == 0:
            continue
        per_class.append(average_precision_score(y_true[:, class_idx], scores[:, class_idx]))
    return float(np.mean(per_class)) if per_class else float("nan")


def fit_and_evaluate_experiment(name, X_train, X_val, X_test):
    best_val_f1 = -np.inf
    best_model = None
    best_params = None
    best_threshold = None
    best_val_scores = None
    candidate_log = []
    total_start = time.perf_counter()

    for C in C_VALUES:
        for gamma in GAMMA_VALUES:
            run_start = time.perf_counter()
            candidate = make_pipeline(
                StandardScaler(),
                OneVsRestClassifier(
                    SVC(
                        kernel="rbf",
                        C=C,
                        gamma=gamma,
                        class_weight="balanced",
                        cache_size=500,
                    ),
                    n_jobs=OVR_N_JOBS,
                ),
            )
            candidate.fit(X_train, Y_train)
            val_scores = candidate.decision_function(X_val)

            config_best_f1 = -np.inf
            config_best_threshold = None
            for threshold in THRESHOLDS:
                predictions = (val_scores >= threshold).astype(np.uint8)
                score = f1_score(Y_val, predictions, average="macro", zero_division=0)
                if score > config_best_f1:
                    config_best_f1 = score
                    config_best_threshold = threshold

            candidate_log.append({
                "C": C,
                "gamma": gamma,
                "best_val_macro_f1": config_best_f1,
                "threshold": config_best_threshold,
                "seconds": time.perf_counter() - run_start,
            })

            if config_best_f1 > best_val_f1:
                best_val_f1 = config_best_f1
                best_model = candidate
                best_params = {"C": C, "gamma": gamma}
                best_threshold = config_best_threshold
                best_val_scores = val_scores

    # Final test set is evaluated only after validation selection is complete.
    test_start = time.perf_counter()
    test_scores = best_model.decision_function(X_test)
    test_predictions = (test_scores >= best_threshold).astype(np.uint8)

    result = {
        "experiment": name,
        "C": best_params["C"],
        "gamma": str(best_params["gamma"]),
        "threshold": best_threshold,
        "val_macro_f1": best_val_f1,
        "val_macro_auroc": safe_macro_auc(Y_val, best_val_scores),
        "val_macro_auprc": safe_macro_ap(Y_val, best_val_scores),
        "test_macro_f1": f1_score(Y_test, test_predictions, average="macro", zero_division=0),
        "test_macro_auroc": safe_macro_auc(Y_test, test_scores),
        "test_macro_auprc": safe_macro_ap(Y_test, test_scores),
        "test_hamming_loss": hamming_loss(Y_test, test_predictions),
        "train_seconds": time.perf_counter() - total_start,
        "test_seconds": time.perf_counter() - test_start,
        "per_class_test_f1": f1_score(Y_test, test_predictions, average=None, zero_division=0),
        "test_predictions": test_predictions,
        "test_scores": test_scores,
        "candidate_log": candidate_log,
    }
    return result, best_model

print("Candidate SVM fits per experiment:", len(C_VALUES) * len(GAMMA_VALUES))
print("Experiments: 1 all-lead baseline + 12 individual leads")
print("Total candidate configurations:", (1 + N_LEADS) * len(C_VALUES) * len(GAMMA_VALUES))

## 6. Run the 13 experiments

This is the computationally expensive section. The feature cache prevents signal processing from being repeated, but each experiment still fits multiple RBF SVM configurations. The small default search grid is deliberate; it can be expanded in the configuration cell once the full pipeline has run successfully.

In [ ]:
results = []
models_to_keep = {}
per_class_rows = []
best_single_lead_name = None
best_single_lead_val_f1 = -np.inf
best_single_lead_model = None
best_single_lead_result = None

experiments = [("12-lead baseline", X_12_train, X_12_val, X_12_test)]
for lead_name in LEAD_NAMES:
    matrix = X_single[lead_name]
    experiments.append((
        f"Lead {lead_name}",
        matrix[train_mask],
        matrix[val_mask],
        matrix[test_mask],
    ))

run_start = time.perf_counter()
for experiment_name, Xtr, Xv, Xte in experiments:
    print(f"\n{'=' * 72}\nTraining: {experiment_name}\nInput shapes: {Xtr.shape}, {Xv.shape}, {Xte.shape}")
    result, fitted_model = fit_and_evaluate_experiment(experiment_name, Xtr, Xv, Xte)
    results.append(result)

    for class_name, class_f1 in zip(CLASSES, result["per_class_test_f1"]):
        per_class_rows.append({"experiment": experiment_name, "class": class_name, "test_f1": class_f1})

    print(
        f"Best validation macro-F1={result['val_macro_f1']:.4f} "
        f"(C={result['C']}, gamma={result['gamma']}, threshold={result['threshold']})"
    )
    print(
        f"Test macro-F1={result['test_macro_f1']:.4f} | "
        f"AUROC={result['test_macro_auroc']:.4f} | "
        f"AUPRC={result['test_macro_auprc']:.4f}"
    )
    print(f"Training + validation-selection time: {result['train_seconds'] / 60:.1f} min")

    if experiment_name == "12-lead baseline":
        models_to_keep[experiment_name] = fitted_model
        models_to_keep["12-lead baseline result"] = result
    else:
        lead_name = experiment_name.replace("Lead ", "", 1)
        if result["val_macro_f1"] > best_single_lead_val_f1:
            best_single_lead_val_f1 = result["val_macro_f1"]
            best_single_lead_name = lead_name
            best_single_lead_model = fitted_model
            best_single_lead_result = result

print("\nTotal time for all experiments: {:.1f} minutes".format((time.perf_counter() - run_start) / 60))
models_to_keep["best single-lead"] = best_single_lead_model
models_to_keep["best single-lead result"] = best_single_lead_result

## 7. Compare leads and identify the best single lead

**Winner selection:** highest validation macro-F1. Validation macro-AUROC is used as a secondary sort key for ties. The fold-10 results are shown for transparent final comparison, but do not determine the selected lead.

In [ ]:
results_df = pd.DataFrame([
    {key: value for key, value in result.items() if key not in {
        "per_class_test_f1", "test_predictions", "test_scores", "candidate_log"
    }}
    for result in results
])

single_lead_df = results_df[results_df["experiment"] != "12-lead baseline"].copy()
single_lead_df = single_lead_df.sort_values(
    ["val_macro_f1", "val_macro_auroc"], ascending=False
).reset_index(drop=True)

comparison_df = results_df.copy()
comparison_df["selection_type"] = np.where(
    comparison_df["experiment"] == "12-lead baseline", "12-lead baseline", "single lead"
)

print("Best single lead selected on validation:", best_single_lead_name)
print("Validation macro-F1:", round(best_single_lead_val_f1, 4))
print("\nSingle-lead ranking (validation-based):")
display(single_lead_df[[
    "experiment", "val_macro_f1", "val_macro_auroc", "test_macro_f1",
    "test_macro_auroc", "test_macro_auprc", "C", "gamma", "threshold"
]])

print("\n12-lead baseline:")
display(results_df[results_df["experiment"] == "12-lead baseline"][[
    "experiment", "val_macro_f1", "test_macro_f1", "test_macro_auroc", "test_macro_auprc", "C", "gamma", "threshold"
]])

## 8. Graphs: validation ranking and final test metrics

In [ ]:
# Figure 1: Validation macro-F1 ranking for single leads + 12-lead baseline.
plot_df = results_df.copy()
plot_df = plot_df.sort_values("val_macro_f1", ascending=True)
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(plot_df["experiment"], plot_df["val_macro_f1"])
ax.set_xlabel("Validation macro-F1 (higher is better)")
ax.set_title("RBF SVM: validation performance by lead configuration")
ax.set_xlim(0, 1)
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

# Figure 2: Final test metrics. Model selection is still validation-based.
metric_cols = ["test_macro_f1", "test_macro_auroc", "test_macro_auprc"]
metric_labels = ["Macro-F1", "Macro-AUROC", "Macro-AUPRC"]
ordered = results_df.sort_values("val_macro_f1", ascending=False)
x = np.arange(len(ordered))
width = 0.25
fig, ax = plt.subplots(figsize=(14, 7))
for idx, (col, label) in enumerate(zip(metric_cols, metric_labels)):
    ax.bar(x + (idx - 1) * width, ordered[col], width, label=label)
ax.set_xticks(x)
ax.set_xticklabels(ordered["experiment"], rotation=45, ha="right")
ax.set_ylabel("Test score")
ax.set_title("Final test metrics by lead configuration")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# Figure 3: Per-superclass test F1 for each single lead.
class_f1_matrix = np.vstack([r["per_class_test_f1"] for r in results if r["experiment"] != "12-lead baseline"])
fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(class_f1_matrix, aspect="auto", vmin=0, vmax=1)
ax.set_yticks(np.arange(N_LEADS))
ax.set_yticklabels(LEAD_NAMES)
ax.set_xticks(np.arange(len(CLASSES)))
ax.set_xticklabels(CLASSES)
ax.set_xlabel("Diagnostic superclass")
ax.set_ylabel("Single ECG lead")
ax.set_title("Per-class test F1 by single lead")
fig.colorbar(im, ax=ax, label="F1")
for row in range(N_LEADS):
    for col in range(len(CLASSES)):
        ax.text(col, row, f"{class_f1_matrix[row, col]:.2f}", ha="center", va="center", fontsize=8)
plt.tight_layout()
plt.show()

## 9. Compare the best single lead against the 12-lead baseline

The selected single lead is the one with the best validation macro-F1. This section compares its final test results directly with the 12-lead baseline and prints per-class metrics.

In [ ]:
baseline_result = next(r for r in results if r["experiment"] == "12-lead baseline")
comparison = pd.DataFrame([
    {
        "model": "12-lead baseline",
        "validation_macro_f1": baseline_result["val_macro_f1"],
        "test_macro_f1": baseline_result["test_macro_f1"],
        "test_macro_auroc": baseline_result["test_macro_auroc"],
        "test_macro_auprc": baseline_result["test_macro_auprc"],
    },
    {
        "model": f"Best single lead ({best_single_lead_name})",
        "validation_macro_f1": best_single_lead_result["val_macro_f1"],
        "test_macro_f1": best_single_lead_result["test_macro_f1"],
        "test_macro_auroc": best_single_lead_result["test_macro_auroc"],
        "test_macro_auprc": best_single_lead_result["test_macro_auprc"],
    },
])
display(comparison)

best_lead_delta = best_single_lead_result["test_macro_f1"] - baseline_result["test_macro_f1"]
print(f"Best single lead by validation: {best_single_lead_name}")
print(f"Test macro-F1 difference (single lead - 12-lead): {best_lead_delta:+.4f}")
print("\nBest single-lead test classification report:")
print(classification_report(
    Y_test,
    best_single_lead_result["test_predictions"],
    target_names=CLASSES,
    zero_division=0,
))
print("12-lead baseline test classification report:")
print(classification_report(
    Y_test,
    baseline_result["test_predictions"],
    target_names=CLASSES,
    zero_division=0,
))

## 10. Multi-label confusion matrices

Each superclass gets its own 2×2 confusion matrix. These are shown for the 12-lead baseline and the validation-selected best single lead.

In [ ]:
def plot_multilabel_confusions(y_true, y_pred, title):
    matrices = multilabel_confusion_matrix(y_true, y_pred)
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for ax, class_name, matrix in zip(axes.flat, CLASSES, matrices):
        ConfusionMatrixDisplay(
            confusion_matrix=matrix,
            display_labels=["Absent", "Present"],
        ).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
        ax.set_title(class_name)
    axes.flat[-1].axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_multilabel_confusions(
    Y_test,
    baseline_result["test_predictions"],
    "12-lead baseline — PTB-XL test fold 10",
)
plot_multilabel_confusions(
    Y_test,
    best_single_lead_result["test_predictions"],
    f"Best single lead ({best_single_lead_name}) — PTB-XL test fold 10",
)